# Demo: Transfer Learning e Fine-Tuning na Prática
**Disciplina:** Inteligência Artificial e Machine Learning Avançado
**Contexto:** Análise de Sentimento de Textos

Neste notebook, vamos desmistificar o processo de Transfer Learning em Modelos de Linguagem (NLP).
Historicamente, treinar um modelo de linguagem do zero exige terabytes de texto e milhões de dólares em GPUs. Hoje, vamos pegar um modelo pré-treinado na Wikipedia (DistilBERT) e adaptá-lo para uma tarefa de Análise de Sentimento.

**Nossa jornada hoje:**
1. Carregar um modelo pré-treinado (A "Mente" Geral).
2. **Feature Extraction:** Usar o modelo apenas como extrator de características (congelando seu conhecimento).
3. **Fine-Tuning:** Descongelar o modelo e ajustá-lo especificamente para nossa tarefa.

In [ ]:
# Instalação das bibliotecas necessárias (descomente se estiver rodando no Google Colab)
# !pip install -q transformers datasets scikit-learn torch

import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Configurando o dispositivo (GPU se disponível, senão CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Rodando em: {device}")

## 1. O Dataset e o Tokenizador
Vamos usar um dataset clássico e leve de análise de sentimento (Rotten Tomatoes - críticas de filmes) contendo frases curtas classificadas como Positivas (1) ou Negativas (0).
O Tokenizador é responsável por transformar nossas palavras nos números que o modelo entende.

In [ ]:
# Carregando um dataset leve para a demonstração (apenas 2000 exemplos para ser rápido)
dataset = load_dataset("rotten_tomatoes")
train_data = dataset["train"].shuffle(seed=42).select(range(2000))
test_data = dataset["test"].shuffle(seed=42).select(range(500))

# Carregando o tokenizador do DistilBERT
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

# Função para tokenizar o texto
def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

# Aplicando a tokenização ao dataset
train_encoded = train_data.map(tokenize, batched=True, batch_size=None)
test_encoded = test_data.map(tokenize, batched=True, batch_size=None)

# Convertendo para tensores PyTorch
train_encoded.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_encoded.set_format("torch", columns=["input_ids", "attention_mask", "label"])

## 2. Abordagem 1: Feature Extraction (Congelando a Rede)
Nesta abordagem de Transfer Learning, nós **não alteramos os pesos** do modelo original. Nós passamos nossos textos por ele, extraímos o "entendimento" do modelo (chamado de *Hidden State* ou *Embedding*) e treinamos um classificador clássico (como Regressão Logística) por cima.

### Exercício 1: Congelando os pesos
Para garantir que o modelo não aprenda nada novo nesta etapa e funcione apenas como um extrator estático, precisamos congelar seus parâmetros iterando sobre eles e desativando o cálculo de gradientes.

In [ ]:
# Carregando o modelo base (sem a camada de classificação final)
model_extractor = AutoModel.from_pretrained(model_ckpt).to(device)

# ==========================================
# EXERCÍCIO 1: Congelando os pesos
# ==========================================
#resposta
# ==========================================

# Função para extrair os embeddings da última camada do modelo
def extract_hidden_states(batch):
    inputs = {k: v.to(device) for k, v in batch.items() if k in tokenizer.model_input_names}
    with torch.no_grad():
        last_hidden_state = model_extractor(**inputs).last_hidden_state
    # Pegamos o vetor correspondente ao token [CLS] (primeira posição), que resume a frase
    return {"hidden_state": last_hidden_state[:, 0].cpu().numpy()}

# Extraindo as features (isso pode levar alguns segundos)
print("Extraindo features...")
train_features = train_encoded.map(extract_hidden_states, batched=True, batch_size=16)
test_features = test_encoded.map(extract_hidden_states, batched=True, batch_size=16)

X_train = np.array(train_features["hidden_state"])
y_train = np.array(train_features["label"])
X_test = np.array(test_features["hidden_state"])
y_test = np.array(test_features["label"])

### Exercício 2: Treinando o "Head" de Classificação
Agora que transformamos nossos textos em matrizes numéricas ricas em contexto (embeddings), precisamos treinar um modelo simples em cima delas.

In [ ]:
# ==========================================
# EXERCÍCIO 2: Treinar Classificador
# ==========================================

# ==========================================

## 3. Abordagem 2: Fine-Tuning (Descongelando a Rede)
Na extração de features, a acurácia foi boa, mas o modelo não adaptou seu "vocabulário" interno para a nossa tarefa específica.

No **Fine-Tuning**, nós substituímos o classificador final e **deixamos o erro (gradiente) fluir por toda a rede**. Assim, o modelo altera seus pesos internos.

### Exercício 3: O Segredo do Fine-Tuning (Taxa de Aprendizado)
Se usarmos uma Taxa de Aprendizado (*Learning Rate*) muito alta, nós destruiremos os pesos que o modelo aprendeu. Por isso, usamos valores muito baixos.

In [ ]:
# Carregamos o modelo já com uma camada de classificação final não-treinada
model_finetune = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2).to(device)

# ==========================================
# EXERCÍCIO 3: Taxa de Aprendizado
learning_rate_escolhido = None
# ==========================================

# Configurações do treinamento
training_args = TrainingArguments(
    output_dir="./resultados",
    learning_rate=learning_rate_escolhido,
    num_train_epochs=2, # Apenas 2 épocas são suficientes com Fine-Tuning!
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch"
)

# Inicializando o Trainer da HuggingFace - PREENCHA OS DADOS
trainer = Trainer(
    model=None,
    args=None,
    train_dataset=None,
    eval_dataset=None
)

print("Iniciando o Fine-Tuning...")
trainer.train()

# Avaliando o resultado
results = trainer.evaluate()
print(f"Resultado do Fine-Tuning: {results}")

In [ ]:
# Faz as predições no dataset de teste
predicoes_brutas = trainer.predict(test_encoded)

# Extrai as classes previstas (0 ou 1)
y_pred_finetune = np.argmax(predicoes_brutas.predictions, axis=-1)

# Pega nas labels reais
y_true_finetune = predicoes_brutas.label_ids

# Calcula a acurácia
acuracia_final = accuracy_score(y_true_finetune, y_pred_finetune)
print(f"Acurácia do Fine-Tuning: {acuracia_final:.4f}")

### Conclusão da Demo
* Com **Feature Extraction**, treinamos um modelo em segundos, pois a rede profunda estava congelada. Excelente quando temos poucos recursos.
* Com **Fine-Tuning**, gastamos mais tempo e processamento, mas o modelo se adaptou profundamente aos dados, geralmente superando o Feature Extraction em acurácia, especialmente em domínios de nicho (como finanças ou direito).